In [1]:
import json
import pandas as pd
import duckdb
import numpy as np
import glob
from pathlib import Path

# Lendo dados

In [2]:
caminhos_arquivos = glob.glob('dados_brutos/json/*.json')

In [3]:
# 1. LISTAS PARA ACUMULAR OS DADOS
# ---------------------------------------------------------
# Dados Gerais
lista_pessoas = []
lista_bancas = []
lista_eventos = []
lista_orientacoes = []
lista_premios = []
lista_projetos = []

# Produção Bibliográfica
lista_bib_artigos = []
lista_bib_livros = []
lista_bib_capitulos = []
lista_bib_trabalhos_congresso = []
lista_bib_resumos_expandidos = []
lista_bib_resumos_congresso = []
lista_bib_artigos_aceitos = []
lista_bib_apresentacoes = []
lista_bib_textos_jornais = []
lista_bib_outras = []

# Produção Técnica
lista_tec_softwares_patente = []
lista_tec_softwares_sem_patente = []
lista_tec_produtos = []
lista_tec_processos = []
lista_tec_trabalhos = []
lista_tec_outras = []
lista_tec_entrevistas = []

# Patentes e Registros
lista_pat_patentes = []
lista_pat_programas = []
lista_pat_desenhos = []

print("Iniciando o processamento dos arquivos JSON...")
#caminhos_arquivos = glob.glob('dados_brutos/*.json')

if not caminhos_arquivos:
    print("ERRO: Nenhum arquivo JSON encontrado na pasta 'dados_brutos/'.")
    exit()

# ---------------------------------------------------------
# 2. EXTRAÇÃO E ACHATAMENTO (FLATTEN)
# ---------------------------------------------------------
for arquivo in caminhos_arquivos:
    with open(arquivo, 'r', encoding='utf-8') as f:
        dados = json.load(f)
        
        id_lattes = dados.get('informacoes_pessoais', {}).get('id_lattes')
        if not id_lattes:
            continue
            
        # --- PESSOAS ---
        df_pessoa = pd.json_normalize(dados['informacoes_pessoais'])
        lista_pessoas.append(df_pessoa)
        
        # --- BANCAS ---
        if 'bancas' in dados:
            for categoria, itens in dados['bancas'].items():
                if itens:
                    df_temp = pd.DataFrame(itens)
                    df_temp['id_lattes'] = id_lattes
                    df_temp['categoria_banca'] = categoria 
                    if 'membros_banca' in df_temp.columns:
                        df_temp['membros_banca'] = df_temp['membros_banca'].astype(str)
                    lista_bancas.append(df_temp)

        # --- EVENTOS ---
        if 'eventos' in dados:
            for categoria, itens in dados['eventos'].items():
                if itens:
                    df_temp = pd.DataFrame(itens)
                    df_temp['id_lattes'] = id_lattes
                    df_temp['categoria_evento'] = categoria
                    lista_eventos.append(df_temp)

        # --- ORIENTAÇÕES ---
        if 'orientacoes' in dados:
            for status, dicionario_niveis in dados['orientacoes'].items():
                for nivel, itens in dicionario_niveis.items():
                    if itens:
                        df_temp = pd.DataFrame(itens)
                        df_temp['id_lattes'] = id_lattes
                        df_temp['status'] = status
                        df_temp['nivel'] = nivel   
                        lista_orientacoes.append(df_temp)

        # --- PRÊMIOS ---
        if 'premios_titulos' in dados and dados['premios_titulos']:
            df_temp = pd.DataFrame(dados['premios_titulos'])
            df_temp['id_lattes'] = id_lattes
            lista_premios.append(df_temp)

        # --- PROJETOS ---
        if 'projetos_pesquisa' in dados and dados['projetos_pesquisa']:
            df_temp = pd.DataFrame(dados['projetos_pesquisa'])
            df_temp['id_lattes'] = id_lattes
            for col in ['descricao', 'integrantes', 'financiadores']:
                if col in df_temp.columns:
                    df_temp[col] = df_temp[col].astype(str)
            lista_projetos.append(df_temp)

        # --- PRODUÇÃO BIBLIOGRÁFICA ---
        prod_bib = dados.get('producao_bibliografica', {})
        def add_to_list(chave, lista_destino):
            itens = prod_bib.get(chave, [])
            if itens:
                df_temp = pd.DataFrame(itens)
                df_temp['id_lattes'] = id_lattes
                lista_destino.append(df_temp)

        add_to_list('artigos_periodicos', lista_bib_artigos)
        add_to_list('livros_publicados', lista_bib_livros)
        add_to_list('capitulos_livros', lista_bib_capitulos)
        add_to_list('trabalhos_completos_congressos', lista_bib_trabalhos_congresso)
        add_to_list('resumos_expandidos', lista_bib_resumos_expandidos)
        add_to_list('resumos_congressos', lista_bib_resumos_congresso)
        add_to_list('artigos_aceitos', lista_bib_artigos_aceitos)
        add_to_list('apresentacoes_trabalhos', lista_bib_apresentacoes)
        add_to_list('textos_jornais', lista_bib_textos_jornais)
        add_to_list('outras_producoes', lista_bib_outras)

        # --- PRODUÇÃO TÉCNICA ---
        prod_tec = dados.get('producao_tecnica', {})
        def add_to_list_tec(chave, lista_destino):
            itens = prod_tec.get(chave, [])
            if itens:
                df_temp = pd.DataFrame(itens)
                df_temp['id_lattes'] = id_lattes
                lista_destino.append(df_temp)

        add_to_list_tec('softwares_com_patente', lista_tec_softwares_patente)
        add_to_list_tec('softwares_sem_patente', lista_tec_softwares_sem_patente)
        add_to_list_tec('produtos_tecnologicos', lista_tec_produtos)
        add_to_list_tec('processos_tecnicas', lista_tec_processos)
        add_to_list_tec('trabalhos_tecnicos', lista_tec_trabalhos)
        add_to_list_tec('outras_producoes_tecnicas', lista_tec_outras)
        add_to_list_tec('entrevistas', lista_tec_entrevistas)

        # --- PATENTES ---
        patentes = dados.get('patentes_registros', {})
        def add_to_list_pat(chave, lista_destino):
            itens = patentes.get(chave, [])
            if itens:
                df_temp = pd.DataFrame(itens)
                df_temp['id_lattes'] = id_lattes
                lista_destino.append(df_temp)

        add_to_list_pat('patentes', lista_pat_patentes)
        add_to_list_pat('programas_computador', lista_pat_programas)
        add_to_list_pat('desenhos_industriais', lista_pat_desenhos)


# ---------------------------------------------------------
# 3. CONSOLIDAÇÃO EM DATAFRAMES EXPLÍCITOS
# ---------------------------------------------------------
print("Consolidando DataFrames...")

def consolidar(lista):
    return pd.concat(lista, ignore_index=True) if lista else pd.DataFrame()

# DataFrames Gerais
df_pessoas = consolidar(lista_pessoas)
df_bancas = consolidar(lista_bancas)
df_eventos = consolidar(lista_eventos)
df_orientacoes = consolidar(lista_orientacoes)
df_premios = consolidar(lista_premios)
df_projetos = consolidar(lista_projetos)

# DataFrames Produção Bibliográfica
df_bib_artigos = consolidar(lista_bib_artigos)
df_bib_livros = consolidar(lista_bib_livros)
df_bib_capitulos = consolidar(lista_bib_capitulos)
df_bib_trab_congresso = consolidar(lista_bib_trabalhos_congresso)
df_bib_resumos_exp = consolidar(lista_bib_resumos_expandidos)
df_bib_resumos_cong = consolidar(lista_bib_resumos_congresso)
df_bib_art_aceitos = consolidar(lista_bib_artigos_aceitos)
df_bib_apresentacoes = consolidar(lista_bib_apresentacoes)
df_bib_textos_jornais = consolidar(lista_bib_textos_jornais)
df_bib_outras = consolidar(lista_bib_outras)

# DataFrames Produção Técnica
df_tec_soft_patente = consolidar(lista_tec_softwares_patente)
df_tec_soft_sem_patente = consolidar(lista_tec_softwares_sem_patente)
df_tec_produtos = consolidar(lista_tec_produtos)
df_tec_processos = consolidar(lista_tec_processos)
df_tec_trabalhos = consolidar(lista_tec_trabalhos)
df_tec_outras = consolidar(lista_tec_outras)
df_tec_entrevistas = consolidar(lista_tec_entrevistas)

# DataFrames Patentes
df_pat_patentes = consolidar(lista_pat_patentes)
df_pat_programas = consolidar(lista_pat_programas)
df_pat_desenhos = consolidar(lista_pat_desenhos)

Iniciando o processamento dos arquivos JSON...
Consolidando DataFrames...


# Tratando dados

## Informações pessoais

In [4]:
df_pessoas.info()

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   id_lattes              50 non-null     str  
 1   nome_completo          50 non-null     str  
 2   nome_citacoes          50 non-null     str  
 3   sexo                   50 non-null     str  
 4   rotulo                 50 non-null     str  
 5   periodo                50 non-null     str  
 6   bolsa_produtividade    50 non-null     str  
 7   endereco_profissional  50 non-null     str  
 8   atualizacao_cv         50 non-null     str  
 9   url                    50 non-null     str  
 10  texto_resumo           50 non-null     str  
dtypes: str(11)
memory usage: 100.3 KB


In [5]:
# 1. Substituir strings vazias e espaços em branco por NaN
# Usa expressão regular para pegar "" ou "   "
df_pessoas.replace(r'^\s*$', np.nan, regex=True, inplace=True)

,id_lattes,nome_completo,nome_citacoes,sexo,rotulo,periodo,bolsa_produtividade,endereco_profissional,atualizacao_cv,url,texto_resumo
0,6067936254653853,Luis Antonio Brasil Kowada,"KOWADA, L. A. B.;KOWADA, LUIS ANTONIO BRASIL;K...",Masculino,* Sem rótulo,NaN,NaN,"Universidade Federal Fluminense, Instituto de ...",12/05/2026,http://lattes.cnpq.br/6067936254653853,Doutor em Engenharia de Sistemas e Computação ...
1,1342924024681635,Igor Monteiro Moraes,"MORAES, I. M.;Moraes, Igor;Moraes, Igor M.;Mor...",Masculino,* Sem rótulo,NaN,NaN,"Universidade Federal Fluminense, Centro Tecnol...",25/04/2026,http://lattes.cnpq.br/1342924024681635,Igor Monteiro Moraes é professor do Departamen...
2,2778417967776066,André Maués Brabo Pereira,"PEREIRA, A. M. B.;Pereira, A.;Pereira, André M...",Masculino,* Sem rótulo,NaN,NaN,"Universidade Federal Fluminense, Instituto de ...",16/01/2026,http://lattes.cnpq.br/2778417967776066,Possui graduação em Engenharia Civil pela Univ...
3,5784860269030800,Antonio Augusto de Aragao Rocha,"ROCHA, A. A. A.;Rocha, Antonio A.A.;Rocha, Ant...",Masculino,* Sem rótulo,NaN,Nível C,"Universidade Federal Fluminense, Centro Tecnol...",13/01/2026,http://lattes.cnpq.br/5784860269030800,Possui graduação em Ciência da Computação pela...
4,3614186131432854,Celso da Cruz Carneiro Ribeiro,"RIBEIRO, C. C. C.;RIBEIRO, C. C.;RIBEIRO, CELS...",Masculino,* Sem rótulo,NaN,Nível 1A,"Universidade Federal Fluminense, Instituto de ...",18/05/2026,http://lattes.cnpq.br/3614186131432854,É membro da Ordem Nacional do Mérito Científic...
5,7279612728721005,Mario Roberto Folhadela Benevides,"BENEVIDES, M. R. F.;Benevides, M. R. F.;Benevi...",Masculino,* Sem rótulo,NaN,Nível 2,"Universidade Federal Fluminense, Instituto de ...",06/03/2026,http://lattes.cnpq.br/7279612728721005,Possui graduação em Engenharia Elétrica pela P...
6,4791589931798048,Esteban Walter Gonzalez Clua,"CLUA, E. W. G.;Clua, Esteban;Gonzalez Clua, Es...",Masculino,* Sem rótulo,NaN,Nível B,"Universidade Federal Fluminense, Centro Tecnol...",30/04/2026,http://lattes.cnpq.br/4791589931798048,professor Titular da Universidade Federal Flum...
7,7793315334001237,Bruno Lopes Vieira,"VIEIRA, Bruno Lopes;LOPES, Bruno;LOPES, B.;Lop...",Masculino,* Sem rótulo,NaN,NaN,"Universidade Federal Fluminense, Centro Tecnol...",12/05/2026,http://lattes.cnpq.br/7793315334001237,Professor na Universidade Federal Fluminense (...
8,0743793296062293,Daniel Cardoso Moraes de Oliveira,"OLIVEIRA, D. C. M.;OLIVEIRA, D.;DE OLIVEIRA, D...",Masculino,* Sem rótulo,NaN,Nível C,"Universidade Federal Fluminense, Centro Tecnol...",12/05/2026,http://lattes.cnpq.br/0743793296062293,Daniel de Oliveira é Professor Associado do In...
9,4603791761884563,João Felipe Nicolaci Pimentel,"PIMENTEL, J. F. N.;PIMENTEL, J. F.;PIMENTEL, J...",Masculino,* Sem rótulo,NaN,NaN,"Universidade Federal Fluminense, Centro Tecnol...",16/01/2026,http://lattes.cnpq.br/4603791761884563,João Felipe Nicolaci Pimentel é professor do I...


In [6]:
# 2. Tratamento da Data de Atualização
# Converte a string '15/10/2025' para um tipo datetime
if 'atualizacao_cv' in df_pessoas.columns:
    df_pessoas['atualizacao_cv'] = pd.to_datetime(
        df_pessoas['atualizacao_cv'], 
        format='%d/%m/%Y', 
        errors='coerce' # Se tiver uma data bizarra (ex: 99/99/9999), vira nulo em vez de quebrar o script
    )

In [7]:
# 3. Limpeza do campo Rótulo
if 'rotulo' in df_pessoas.columns:
    # Remove o asterisco e espaços em branco nas pontas
    df_pessoas['rotulo'] = df_pessoas['rotulo'].str.replace('*', '', regex=False).str.strip()
    # Se o rótulo ficou "Sem rótulo", transforma em nulo verdadeiro
    df_pessoas['rotulo'] = df_pessoas['rotulo'].replace('Sem rótulo', np.nan)

In [8]:
# 5. Garantia de Tipagem da Chave Primária e Textos Longos
df_pessoas['id_lattes'] = df_pessoas['id_lattes'].astype(str)

In [9]:
if 'texto_resumo' in df_pessoas.columns:
    # Tira quebras de linha e espaços duplos extras no começo e no fim do resumo
    df_pessoas['texto_resumo'] = df_pessoas['texto_resumo'].str.strip()

In [10]:
if 'texto_resumo' in df_pessoas.columns:
    # Tira quebras de linha e espaços duplos extras no começo e no fim do resumo
    df_pessoas['texto_resumo'] = df_pessoas['texto_resumo'].str.strip()

In [11]:
df_pessoas.head()

,id_lattes,nome_completo,nome_citacoes,sexo,rotulo,periodo,bolsa_produtividade,endereco_profissional,atualizacao_cv,url,texto_resumo
0,6067936254653853,Luis Antonio Brasil Kowada,"KOWADA, L. A. B.;KOWADA, LUIS ANTONIO BRASIL;K...",Masculino,NaN,NaN,NaN,"Universidade Federal Fluminense, Instituto de ...",2026-05-12,http://lattes.cnpq.br/6067936254653853,Doutor em Engenharia de Sistemas e Computação ...
1,1342924024681635,Igor Monteiro Moraes,"MORAES, I. M.;Moraes, Igor;Moraes, Igor M.;Mor...",Masculino,NaN,NaN,NaN,"Universidade Federal Fluminense, Centro Tecnol...",2026-04-25,http://lattes.cnpq.br/1342924024681635,Igor Monteiro Moraes é professor do Departamen...
2,2778417967776066,André Maués Brabo Pereira,"PEREIRA, A. M. B.;Pereira, A.;Pereira, André M...",Masculino,NaN,NaN,NaN,"Universidade Federal Fluminense, Instituto de ...",2026-01-16,http://lattes.cnpq.br/2778417967776066,Possui graduação em Engenharia Civil pela Univ...
3,5784860269030800,Antonio Augusto de Aragao Rocha,"ROCHA, A. A. A.;Rocha, Antonio A.A.;Rocha, Ant...",Masculino,NaN,NaN,Nível C,"Universidade Federal Fluminense, Centro Tecnol...",2026-01-13,http://lattes.cnpq.br/5784860269030800,Possui graduação em Ciência da Computação pela...
4,3614186131432854,Celso da Cruz Carneiro Ribeiro,"RIBEIRO, C. C. C.;RIBEIRO, C. C.;RIBEIRO, CELS...",Masculino,NaN,NaN,Nível 1A,"Universidade Federal Fluminense, Instituto de ...",2026-05-18,http://lattes.cnpq.br/3614186131432854,É membro da Ordem Nacional do Mérito Científic...


## Tratando orientações

In [12]:
df_orientacoes.head()

,titulo,ano_inicio,orientando,tipo_trabalho,instituicao,curso,id_lattes,status,nivel,ano_conclusao
0,Deterministic Energy-Based Models for NP-Compl...,2026,Marcelo Nestor da Silva,Tese,Universidade Federal Fluminense,,6067936254653853,em_andamento,doutorado,NaN
1,SIMULAÇÃO QUÂNTICA PARA APLICAÇÕES QML,2026,Lucas Amaral dos Santos Barroso Leite,Tese,Universidade Federal Fluminense,,6067936254653853,em_andamento,doutorado,NaN
2,Algoritmos quânticos para Logaritmo Discreto,2020,Fábio Gomes dos Santos,Tese,Universidade Federal Fluminense,,6067936254653853,em_andamento,doutorado,NaN
3,Algoritmos Quanticos de Otimização,2026,Yohanna Dvorak Mendes,Dissertação,Universidade Federal Fluminense,,6067936254653853,em_andamento,mestrado,NaN
4,Criptografia homomórfica,2024,Victor Faria Fernandes,Dissertação,Universidade Federal Fluminense,,6067936254653853,em_andamento,mestrado,NaN


In [13]:
print(df_orientacoes['tipo_trabalho'].value_counts())
print(df_orientacoes['status'].value_counts())
print(df_orientacoes['nivel'].value_counts())
print(df_orientacoes['curso'].value_counts())

tipo_trabalho
Trabalho de Conclusão de Curso    1714
Dissertação                       1085
Tese                               559
                                   135
Trabalho                            76
Iniciação científica                33
Monografia                          14
Name: count, dtype: int64
status
concluidas      3235
em_andamento     381
Name: count, dtype: int64
nivel
tcc                     1231
mestrado                1095
doutorado                560
iniciacao_cientifica     544
outros                   102
pos_doutorado             60
especializacao            24
Name: count, dtype: int64
curso
    3616
Name: count, dtype: int64


In [14]:
df_orientacoes = df_orientacoes.rename(columns={'titulo': 'titulo_trabalho'})

In [15]:
import pandas as pd

print("Iniciando o tratamento da tabela de orientações...")

# 1. Tratamento de Strings: Remove espaços duplos e quebras de linha escondidas
colunas_texto = ['titulo_trabalho', 'orientando', 'tipo_trabalho', 'instituicao', 'curso']
for col in colunas_texto:
    df_orientacoes[col] = df_orientacoes[col].astype(str).str.strip()
    # Se, após limpar os espaços, o campo ficar vazio ou 'nan', preenche com 'Não informado'
    df_orientacoes[col] = df_orientacoes[col].replace({'': 'Não informado', 'nan': 'Não informado', 'None': 'Não informado'})

# 2. Conversão segura de anos (de float para Int64 com suporte a Nulo)
df_orientacoes['ano_conclusao'] = df_orientacoes['ano_conclusao'].astype('Int64')

# 3. Melhoria estética na coluna 'Nível' para os gráficos do Streamlit
mapeamento_nivel = {
    'mestrado': 'Mestrado',
    'doutorado': 'Doutorado',
    'tcc': 'TCC',
    'iniciacao_cientifica': 'Iniciação Científica',
    'pos_doutorado': 'Pós-Doutorado',
    'especializacao': 'Especialização',
    'outros': 'Outros'
}
df_orientacoes['nivel'] = df_orientacoes['nivel'].map(mapeamento_nivel).fillna(df_orientacoes['nivel'])

# 4. Melhoria estética na coluna 'Status' para os gráficos
mapeamento_status = {
    'concluidas': 'Concluída',
    'em_andamento': 'Em Andamento'
}
df_orientacoes['status'] = df_orientacoes['status'].map(mapeamento_status).fillna(df_orientacoes['status'])

print("\nTratamento concluído com sucesso! Verifique a nova estrutura:")
df_orientacoes.info()

print("\nAmostra dos dados tratados:")
display(df_orientacoes[['orientando', 'nivel', 'status', 'ano_conclusao']].head())

Iniciando o tratamento da tabela de orientações...

Tratamento concluído com sucesso! Verifique a nova estrutura:
<class 'pandas.DataFrame'>
RangeIndex: 3616 entries, 0 to 3615
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   titulo_trabalho  3616 non-null   str  
 1   ano_inicio       3616 non-null   int64
 2   orientando       3616 non-null   str  
 3   tipo_trabalho    3616 non-null   str  
 4   instituicao      3616 non-null   str  
 5   curso            3616 non-null   str  
 6   id_lattes        3616 non-null   str  
 7   status           3616 non-null   str  
 8   nivel            3616 non-null   str  
 9   ano_conclusao    3235 non-null   Int64
dtypes: Int64(1), int64(1), str(8)
memory usage: 1.0 MB

Amostra dos dados tratados:


,orientando,nivel,status,ano_conclusao
0,Marcelo Nestor da Silva,Doutorado,Em Andamento,<NA>
1,Lucas Amaral dos Santos Barroso Leite,Doutorado,Em Andamento,<NA>
2,Fábio Gomes dos Santos,Doutorado,Em Andamento,<NA>
3,Yohanna Dvorak Mendes,Mestrado,Em Andamento,<NA>
4,Victor Faria Fernandes,Mestrado,Em Andamento,<NA>


## Informações acerca de periódicos publicados

In [16]:
df_bib_artigos.head()

,titulo,ano,autores,revista,volume,numero,paginas,issn,doi,qualis,id_lattes
0,Improved Biclique Cryptanalysis of the Lightwe...,2026,"DE CARVALHO, GABRIEL ; KOWADA, LUIS",JOURNAL OF THE BRAZILIAN COMPUTER SOCIETY (ONL...,32,,539-554,1678-4804,http://dx.doi.org/10.5753/jbcs.2026.5390,,6067936254653853
1,Quantum kernel and HHL-based support vector ma...,2026,"PINHEIRO, GABRIELA ; SLABBERT, DONOVAN M. ; KO...",EPJ QUANTUM TECHNOLOGY,2026,,1-18,2662-4400,http://dx.doi.org/10.1140/epjqt/s40507-026-004...,,6067936254653853
2,USING SIMULATIONS TO VALIDATE IMPROVEMENTS OVE...,2025,"SANTOS, FÁBIO G. DOS ; KOWADA, LUIS A. B.",TECNOLOGIA & CULTURA (CEFET/RJ),2025,,30-34,1414-8498,,,6067936254653853
3,"HHL: Estado da Arte, Limitações e Melhorias",2025,"LEITE, L. A. S. ; KOWADA, LUIS A. B.",TECNOLOGIA & CULTURA (CEFET/RJ),2025,,35-39,1414-8498,,,6067936254653853
4,Analysis of the Quantum Algorithm HHL for the ...,2025,"PINHEIRO, G. ; KOWADA, LUIS A. B.",TECNOLOGIA & CULTURA (CEFET/RJ),2025,,59-62,1414-8498,,,6067936254653853


In [17]:
print("Aplicando tratamentos na tabela 'bib_artigos'...")

if not df_bib_artigos.empty:
    
    # 1. Transformar strings vazias ou só com espaços em nulos reais (NaN)
    df_bib_artigos.replace(r'^\s*$', np.nan, regex=True, inplace=True)
    
    # 2. Tratamento da Revista (Periódico)
    if 'revista' in df_bib_artigos.columns:
        # Força maiúsculo e remove espaços extras no início e no fim
        df_bib_artigos['revista'] = df_bib_artigos['revista'].str.upper().str.strip()

    # 4. Tratamento do Ano (Garantir que seja número inteiro)
    if 'ano' in df_bib_artigos.columns:
        # errors='coerce' transforma erros (ex: "Sem ano") em NaN
        # Int64 é o tipo inteiro do Pandas que aceita valores nulos
        df_bib_artigos['ano'] = pd.to_numeric(df_bib_artigos['ano'], errors='coerce').astype('Int64')

    # 5. Tratamento de Título, DOI e ISSN (Apenas remover espaços ocultos)
    colunas_texto = ['titulo', 'doi', 'issn', 'volume', 'numero', 'paginas']
    for col in colunas_texto:
        if col in df_bib_artigos.columns:
            df_bib_artigos[col] = df_bib_artigos[col].str.strip()

    # 6. Garantir tipagem da chave primária
    df_bib_artigos['id_lattes'] = df_bib_artigos['id_lattes'].astype(str)

print("Tratamento da tabela 'bib_artigos' concluído!")
display(df_bib_artigos[['ano', 'revista', 'doi']].head())

Aplicando tratamentos na tabela 'bib_artigos'...
Tratamento da tabela 'bib_artigos' concluído!


,ano,revista,doi
0,2026,JOURNAL OF THE BRAZILIAN COMPUTER SOCIETY (ONL...,http://dx.doi.org/10.5753/jbcs.2026.5390
1,2026,EPJ QUANTUM TECHNOLOGY,http://dx.doi.org/10.1140/epjqt/s40507-026-004...
2,2025,TECNOLOGIA & CULTURA (CEFET/RJ),NaN
3,2025,TECNOLOGIA & CULTURA (CEFET/RJ),NaN
4,2025,TECNOLOGIA & CULTURA (CEFET/RJ),NaN


In [18]:
df_bib_artigos.info()

<class 'pandas.DataFrame'>
RangeIndex: 2018 entries, 0 to 2017
Data columns (total 11 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   titulo     2018 non-null   str  
 1   ano        2018 non-null   Int64
 2   autores    2018 non-null   str  
 3   revista    2018 non-null   str  
 4   volume     1988 non-null   str  
 5   numero     103 non-null    str  
 6   paginas    2017 non-null   str  
 7   issn       1998 non-null   str  
 8   doi        1675 non-null   str  
 9   qualis     0 non-null      str  
 10  id_lattes  2018 non-null   str  
dtypes: Int64(1), str(10)
memory usage: 717.2 KB


## Tratando informações de eventos

In [19]:
df_bib_trab_congresso.head()

,titulo,ano,doi,autores,evento,cidade,paginas,isbn,id_lattes
0,All Direct Product C5 × Kn Graphs Are Type 1,2025,http://dx.doi.org/10.5540/03.2026.012.01.0243,"CASTONGUAY, DIANE ; FIGUEIREDO, CELINA M. H. D...",CNMAC 2025,,,,6067936254653853
1,Improving Qiskit strategies for circuit synthe...,2025,http://dx.doi.org/10.5753/semish.2025.9256,"LIMA, RAPHAEL B. F. ; Kowada, Luis Antonio B. ...",Seminário Integrado de Software e Hardware,,573,,6067936254653853
2,Automation of the Quantum Algorithm HHL for im...,2024,http://dx.doi.org/10.5753/wqunets.2024.2861,"PINHEIRO, G. ; KOWADA, L. A. B.",I Workshop de Redes Quânticas,,13-18,,6067936254653853
3,The Best Biclique Cryptanalysis of the Lightwe...,2024,http://dx.doi.org/10.5753/sbseg.2024.241733,"CARVALHO, G. C. DE ; KOWADA, L. A. B.",Simpósio Brasileiro de Segurança da Informação...,,586-599,,6067936254653853
4,Implementing homomorphic encryption for image ...,2024,,"FERNANDES, V. ; BERNARDINO, R. ; KOWADA, L.",IEEE International Conference on Cloud Networking,,,,6067936254653853


In [20]:
df_bib_trab_congresso.info()

<class 'pandas.DataFrame'>
RangeIndex: 4461 entries, 0 to 4460
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   titulo     4461 non-null   str  
 1   ano        4461 non-null   int64
 2   doi        4461 non-null   str  
 3   autores    4461 non-null   str  
 4   evento     4461 non-null   str  
 5   cidade     4461 non-null   str  
 6   paginas    4461 non-null   str  
 7   isbn       4461 non-null   str  
 8   id_lattes  4461 non-null   str  
dtypes: int64(1), str(8)
memory usage: 1.4 MB


In [21]:
print("Aplicando tratamentos na tabela 'df_bib_trab_congresso'...")

if not df_bib_trab_congresso.empty:
    
    # 1. Transformar strings vazias ou só com espaços em nulos reais (NaN)
    df_bib_trab_congresso.replace(r'^\s*$', np.nan, regex=True, inplace=True)
    
    # 2. Tratamento da Revista (Periódico)
    if 'evento' in df_bib_trab_congresso.columns:
        # Força maiúsculo e remove espaços extras no início e no fim
        df_bib_trab_congresso['evento'] = df_bib_trab_congresso['evento'].str.upper().str.strip()

    # 4. Tratamento do Ano (Garantir que seja número inteiro)
    if 'ano' in df_bib_trab_congresso.columns:
        # errors='coerce' transforma erros (ex: "Sem ano") em NaN
        # Int64 é o tipo inteiro do Pandas que aceita valores nulos
        df_bib_trab_congresso['ano'] = pd.to_numeric(df_bib_trab_congresso['ano'], errors='coerce').astype('Int64')

    # 5. Tratamento de Título, DOI e ISSN (Apenas remover espaços ocultos)
    colunas_texto = ['titulo', 'doi', 'isbn', 'paginas']
    for col in colunas_texto:
        if col in df_bib_trab_congresso.columns:
            df_bib_trab_congresso[col] = df_bib_trab_congresso[col].str.strip()

    # 6. Garantir tipagem da chave primária
    df_bib_trab_congresso['id_lattes'] = df_bib_trab_congresso['id_lattes'].astype(str)

print("Tratamento da tabela 'bib_artigos' concluído!")
display(df_bib_trab_congresso[['ano', 'evento', 'doi']].head())

Aplicando tratamentos na tabela 'df_bib_trab_congresso'...
Tratamento da tabela 'bib_artigos' concluído!


,ano,evento,doi
0,2025,CNMAC 2025,http://dx.doi.org/10.5540/03.2026.012.01.0243
1,2025,SEMINÁRIO INTEGRADO DE SOFTWARE E HARDWARE,http://dx.doi.org/10.5753/semish.2025.9256
2,2024,I WORKSHOP DE REDES QUÂNTICAS,http://dx.doi.org/10.5753/wqunets.2024.2861
3,2024,SIMPÓSIO BRASILEIRO DE SEGURANÇA DA INFORMAÇÃO...,http://dx.doi.org/10.5753/sbseg.2024.241733
4,2024,IEEE INTERNATIONAL CONFERENCE ON CLOUD NETWORKING,NaN


In [22]:
df_bib_trab_congresso.head(3)

,titulo,ano,doi,autores,evento,cidade,paginas,isbn,id_lattes
0,All Direct Product C5 × Kn Graphs Are Type 1,2025,http://dx.doi.org/10.5540/03.2026.012.01.0243,"CASTONGUAY, DIANE ; FIGUEIREDO, CELINA M. H. D...",CNMAC 2025,NaN,NaN,NaN,6067936254653853
1,Improving Qiskit strategies for circuit synthe...,2025,http://dx.doi.org/10.5753/semish.2025.9256,"LIMA, RAPHAEL B. F. ; Kowada, Luis Antonio B. ...",SEMINÁRIO INTEGRADO DE SOFTWARE E HARDWARE,NaN,573,NaN,6067936254653853
2,Automation of the Quantum Algorithm HHL for im...,2024,http://dx.doi.org/10.5753/wqunets.2024.2861,"PINHEIRO, G. ; KOWADA, L. A. B.",I WORKSHOP DE REDES QUÂNTICAS,NaN,13-18,NaN,6067936254653853


## Tratando informações de classificação

### Database de periódicos

In [23]:
import pandas as pd
import numpy as np

print("Épata 1: Carregando e preparando a base completa da Scopus...")
# Carrega o ficheiro original intacto da Scopus
df_scopus_raw = pd.read_excel('periodicos_percentil.xlsx')

# Limpeza de segurança padrão nas chaves de cruzamento
df_scopus_raw['Title'] = df_scopus_raw['Title'].str.upper().str.strip()
df_scopus_raw['E-ISSN'] = df_scopus_raw['E-ISSN'].astype(str).str.replace('-', '', regex=False).str.strip()

# Mapeia TODOS os títulos/ISSNs que possuem alguma subárea de computação
mask_comput = df_scopus_raw['Scopus Sub-Subject Area'].str.contains('Comput', case=False, na=False)
titulos_computacao = set(df_scopus_raw[mask_comput]['Title'].unique())
issns_computacao = set(df_scopus_raw[mask_comput]['E-ISSN'].unique())

# Ordena pelo Percentile (do maior para o menor) e remove duplicados de títulos.
# Isto garante que cada revista apareça apenas uma vez, retendo as informações da linha de maior percentil.
df_scopus_unicos = df_scopus_raw.sort_values(by='Percentile', ascending=False)
df_scopus_unicos = df_scopus_unicos.drop_duplicates(subset=['Title'], keep='first').copy()

print("Etapa 2: Preparando a base de artigos do Lattes...")
# Limpeza de segurança padrão nas chaves dos artigos
df_bib_artigos['revista'] = df_bib_artigos['revista'].str.upper().str.strip()
df_bib_artigos['issn'] = df_bib_artigos['issn'].astype(str).str.replace('-', '', regex=False).str.strip()

print("Etapa 3: Realizando o cruzamento exato (Match)...")
# Seleciona apenas as colunas da Scopus que solicitou para o join
colunas_scopus = [
    'Scopus Source ID', 'Title', 'Percentile', 
    'Scopus ASJC Code (Sub-subject Area)', 'Scopus Sub-Subject Area', 'E-ISSN'
]
df_scopus_filtro = df_scopus_unicos[colunas_scopus]

# Match pelo Nome da Revista
df_match_nome = pd.merge(df_bib_artigos, df_scopus_filtro, left_on='revista', right_on='Title', how='inner')

# Match pelo ISSN
df_match_issn = pd.merge(df_bib_artigos, df_scopus_filtro, left_on='issn', right_on='E-ISSN', how='inner')

# Consolida os sucessos e remove possíveis duplicados (artigos que deram match por ambos os critérios)
df_sucessos = pd.concat([df_match_nome, df_match_issn], ignore_index=True)
df_sucessos = df_sucessos.drop_duplicates(subset=['titulo', 'id_lattes']).copy()

# Cria a coluna booleana baseada nas listas completas que mapeamos no início
df_sucessos['Computation Area'] = df_sucessos['Title'].isin(titulos_computacao) | df_sucessos['E-ISSN'].isin(issns_computacao)

print("Etapa 4: Isolando e tratando os artigos sem correspondência (No Match)...")
# Filtra os artigos que ficaram de fora do grupo de sucessos
artigos_com_match = set(df_sucessos['titulo'] + df_sucessos['id_lattes'])
df_falhas = df_bib_artigos[~(df_bib_artigos['titulo'] + df_bib_artigos['id_lattes']).isin(artigos_com_match)].copy()

# Preenche os atributos padrão solicitados para os artigos não encontrados
df_falhas['Scopus Source ID'] = pd.NA
df_falhas['Title'] = pd.NA
df_falhas['Percentile'] = 0  # <--- Solicitado: 0 para não encontrados
df_falhas['Scopus ASJC Code (Sub-subject Area)'] = pd.NA
df_falhas['Scopus Sub-Subject Area'] = pd.NA
df_falhas['E-ISSN'] = pd.NA
df_falhas['Computation Area'] = False  # <--- Solicitado: False para não encontrados

print("Etapa 5: Consolidando a tabela final...")
# Empilha os dois blocos de volta
df_artigos_final = pd.concat([df_sucessos, df_falhas], ignore_index=True)

# Garante a tipagem ideal das novas colunas
df_artigos_final['Percentile'] = df_artigos_final['Percentile'].astype(int)
df_artigos_final['Computation Area'] = df_artigos_final['Computation Area'].astype(bool)

print("\n🚀 Tabela final construída com sucesso!")
print(f"Total de linhas: {len(df_artigos_final)}")

# Define a ordem de colunas ideal exibindo primeiro os dados do artigo e depois as métricas solicitadas
colunas_finais = [
    'id_lattes', 'titulo', 'revista', 'ano', # Dados do Artigo
    'Scopus Source ID', 'Title', 'Percentile', 
    'Scopus ASJC Code (Sub-subject Area)', 'Scopus Sub-Subject Area', 'E-ISSN', # Dados Scopus
    'Computation Area' # Indicador criado
]
df_artigos_final = df_artigos_final[colunas_finais]

# Mostra uma amostra mista contendo sucessos e falhas para validação
display(df_artigos_final.sample(10))

Épata 1: Carregando e preparando a base completa da Scopus...
Etapa 2: Preparando a base de artigos do Lattes...
Etapa 3: Realizando o cruzamento exato (Match)...
Etapa 4: Isolando e tratando os artigos sem correspondência (No Match)...
Etapa 5: Consolidando a tabela final...

🚀 Tabela final construída com sucesso!
Total de linhas: 2015


,id_lattes,titulo,revista,ano,Scopus Source ID,Title,Percentile,Scopus ASJC Code (Sub-subject Area),Scopus Sub-Subject Area,E-ISSN,Computation Area
126,3614186131432854,A distributed and hierarchical strategy for au...,INTERNATIONAL TRANSACTIONS IN OPERATIONAL RESE...,2011,9700153238,INTERNATIONAL TRANSACTIONS IN OPERATIONAL RESE...,80,1403,Business and International Management,14753995,True
363,1980219310993467,Enabling FEM-based absolute permeability estim...,COMPUTER METHODS IN APPLIED MECHANICS AND ENGI...,2025,18158,COMPUTER METHODS IN APPLIED MECHANICS AND ENGI...,98,2206,Computational Mechanics,NaN,True
1381,7279612728721005,Propositional dynamic logic for Petri Nets,LOGIC JOURNAL OF THE IGPL (ONLINE),2014,<NA>,<NA>,0,<NA>,<NA>,<NA>,False
207,7279612728721005,Using modal logics to express and check global...,LOGIC JOURNAL OF THE IGPL,2009,5000154601,LOGIC JOURNAL OF THE IGPL,70,2609,Logic,13689894,False
24,1342924024681635,FITS: A flexible virtual network testbed archi...,COMPUTER NETWORKS,2014,26811,COMPUTER NETWORKS,87,1705,Computer Networks and Communications,NaN,True
1777,9171815778534257,A Hybrid Heuristic based on Iterated Local Sea...,ELECTRONIC NOTES IN DISCRETE MATHEMATICS,2016,<NA>,<NA>,0,<NA>,<NA>,<NA>,False
1287,5386282151810710,An approach based on the domain perspective to...,SOFTWARE & SYSTEMS MODELING,2017,144641,SOFTWARE AND SYSTEMS MODELING,89,2611,Modeling and Simulation,16191374,False
1464,5601388085745497,Transfer Learning and Fine Tuning in breast ma...,"ADVANCES IN SCIENCE, TECHNOLOGY AND ENGINEERIN...",2020,<NA>,<NA>,0,<NA>,<NA>,<NA>,False
612,8795680989708219,Towards Optimal Static Task Scheduling for Rea...,INTERNATIONAL JOURNAL OF HIGH PERFORMANCE COMP...,2003,19268,INTERNATIONAL JOURNAL OF HIGH PERFORMANCE COMP...,81,2614,Theoretical Computer Science,17412846,True
745,4616848792501359,Using Conventional Cameras as Sensors for Esti...,SENSORS,2022,130124,SENSORS,88,3105,Instrumentation,14248220,False


In [24]:
# Cria a coluna indicando se o match ocorreu de forma adequada
# O método .notna() retorna True se existe um ID da Scopus, e False se for nulo (<NA>)
df_artigos_final['match_adequado'] = df_artigos_final['Scopus Source ID'].notna()

# Atualizando a lista de colunas para exibir essa nova no começo
colunas_finais = [
    'id_lattes', 'titulo', 'revista', 'ano', 'match_adequado', # <- Nova coluna aqui
    'Scopus Source ID', 'Title', 'Percentile', 
    'Scopus ASJC Code (Sub-subject Area)', 'Scopus Sub-Subject Area', 'E-ISSN', 
    'Computation Area'
]
df_artigos_final = df_artigos_final[colunas_finais]

print("\nColuna 'match_adequado' adicionada com sucesso!")
display(df_artigos_final[['titulo', 'revista', 'match_adequado', 'Percentile']].sample(10))


Coluna 'match_adequado' adicionada com sucesso!


,titulo,revista,match_adequado,Percentile
1995,Experience of Using GPU in Power Flow Computation,CONCURRENCY AND COMPUTATION-PRACTICE & EXPERIENCE,False,0
536,Matching preclusion number in Cartesian produc...,ARS COMBINATORIA,True,4
1533,Designing screen layout in multimedia applicat...,RAIRO-OPERATIONS RESEARCH,False,0
99,Compact formulations and an iterated local sea...,EUROPEAN JOURNAL OF OPERATIONAL RESEARCH,True,98
333,Automated recognition of the pericardium conto...,COMPUTERS IN BIOLOGY AND MEDICINE,True,94
868,Power System Probabilistic Reliability Assessm...,IEEE TRANSACTIONS ON POWER SYSTEMS,True,96
1211,Analyzing the adoption of database management ...,EMPIRICAL SOFTWARE ENGINEERING (DORDRECHT. ONL...,True,79
509,On clique-inverse graphs of graphs with bounde...,JOURNAL OF GRAPH THEORY,True,70
50,Ontogeny of divided vascular cylinders in Serj...,IAWA JOURNAL,True,72
1894,BERTweet.BR: a pre-trained language model for ...,NEURAL COMPUTING & APPLICATIONS,False,0


In [25]:
df_artigos_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 2015 entries, 0 to 2014
Data columns (total 12 columns):
 #   Column                               Non-Null Count  Dtype 
---  ------                               --------------  ----- 
 0   id_lattes                            2015 non-null   str   
 1   titulo                               2015 non-null   str   
 2   revista                              2015 non-null   str   
 3   ano                                  2015 non-null   Int64 
 4   match_adequado                       2015 non-null   bool  
 5   Scopus Source ID                     1301 non-null   object
 6   Title                                1301 non-null   object
 7   Percentile                           2015 non-null   int64 
 8   Scopus ASJC Code (Sub-subject Area)  1301 non-null   object
 9   Scopus Sub-Subject Area              1301 non-null   object
 10  E-ISSN                               758 non-null    object
 11  Computation Area                     2015 non-null   b

In [26]:
print("Renomeando as colunas do DataFrame final...")

# Dicionário com o mapeamento "Nome Antigo" : "Nome Novo"
mapeamento_colunas = {
    'titulo': 'titulo_artigo',
    'revista': 'titulo_revista_lattes',
    'ano': 'ano_pub',
    'Scopus Source ID': 'id_scopus',
    'Title': 'titulo_revista_scopus',
    'Percentile': 'maior_percentil',
    'Scopus ASJC Code (Sub-subject Area)': 'codigo_area_maior_percentil',
    'Scopus Sub-Subject Area': 'area_maior_percentil',
    'E-ISSN': 'issn',
    'Computation Area': 'computation_area'
}

# Aplica a renomeação diretamente no dataframe
df_artigos_final.rename(columns=mapeamento_colunas, inplace=True)

print("Colunas renomeadas com sucesso! Nova estrutura:")
df_artigos_final.info()

Renomeando as colunas do DataFrame final...
Colunas renomeadas com sucesso! Nova estrutura:
<class 'pandas.DataFrame'>
RangeIndex: 2015 entries, 0 to 2014
Data columns (total 12 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   id_lattes                    2015 non-null   str   
 1   titulo_artigo                2015 non-null   str   
 2   titulo_revista_lattes        2015 non-null   str   
 3   ano_pub                      2015 non-null   Int64 
 4   match_adequado               2015 non-null   bool  
 5   id_scopus                    1301 non-null   object
 6   titulo_revista_scopus        1301 non-null   object
 7   maior_percentil              2015 non-null   int64 
 8   codigo_area_maior_percentil  1301 non-null   object
 9   area_maior_percentil         1301 non-null   object
 10  issn                         758 non-null    object
 11  computation_area             2015 non-null   bool  
dtypes: Int64(

### Database de conferências

In [27]:
df_google_raw = pd.read_excel('eventos_classificados.xlsx')

In [28]:
df_google_raw.head(3)

,Sigla,Nome do evento,Estrato
0,AAAI,AAAI Conference on Artificial Intelligence,A1
1,AAMAS,International Conference on Autonomous Agents ...,A1
2,ACCV,Asian Conference on Computer Vision,A1


In [29]:
df_google_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 781 entries, 0 to 780
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Sigla           781 non-null    str  
 1   Nome do evento  781 non-null    str  
 2   Estrato         781 non-null    str  
dtypes: str(3)
memory usage: 68.5 KB


In [30]:
df_google_raw['Estrato'].value_counts()

Estrato
A3    171
A4    134
A1    110
B4     90
A2     86
B1     78
B2     60
B3     52
Name: count, dtype: int64

In [31]:
print("Substituindo as classificações na coluna 'Estrato'...")

# 1. Cria o dicionário de substituição ('Valor Antigo': 'Valor Novo')
mapeamento_estratos = {
    'B1': 'A5',
    'B2': 'A6',
    'B3': 'A7',
    'B4': 'A8'
}

# 2. Aplica a substituição apenas na coluna 'Estrato'
df_google_raw['Estrato'] = df_google_raw['Estrato'].replace(mapeamento_estratos)

# 3. Verifica o resultado para garantir que deu certo
print("\nNova distribuição de Estratos:")
display(df_google_raw['Estrato'].value_counts())

Substituindo as classificações na coluna 'Estrato'...

Nova distribuição de Estratos:


Estrato
A3    171
A4    134
A1    110
A8     90
A2     86
A5     78
A6     60
A7     52
Name: count, dtype: int64

In [32]:
# Transforma a coluna 'Nome do evento' para letras maiúsculas
df_google_raw['Nome do evento'] = df_google_raw['Nome do evento'].str.upper()

# Exibe as primeiras linhas para confirmar a alteração
display(df_google_raw.head())

,Sigla,Nome do evento,Estrato
0,AAAI,AAAI CONFERENCE ON ARTIFICIAL INTELLIGENCE,A1
1,AAMAS,INTERNATIONAL CONFERENCE ON AUTONOMOUS AGENTS ...,A1
2,ACCV,ASIAN CONFERENCE ON COMPUTER VISION,A1
3,ACII,INTERNATIONAL CONFERENCE ON AFFECTIVE COMPUTIN...,A2
4,ACISP,AUSTRALASIAN CONFERENCE ON INFORMATION SECURIT...,A4


In [33]:
import pandas as pd
import re

print("1. Preparando os dados para o cruzamento...")
# Criar coluna limpa no dataframe do Lattes para não perder o dado original
df_bib_trab_congresso['evento_limpo'] = df_bib_trab_congresso['evento'].astype(str).str.upper().str.strip()

# Limpeza no dataframe do Google
df_google_raw['Nome do evento'] = df_google_raw['Nome do evento'].str.upper().str.strip()
df_google_raw['Sigla'] = df_google_raw['Sigla'].str.upper().str.strip()

print("2. Tentando Match Exato pelo Nome do Evento...")
# Cruzamento direto: evento == Nome do evento
df_match_exato = pd.merge(
    df_bib_trab_congresso, 
    df_google_raw, 
    left_on='evento_limpo', 
    right_on='Nome do evento', 
    how='inner'
)
df_match_exato['tipo_match'] = 'Exato'

print("3. Isolando os artigos não encontrados para a busca por Sigla...")
# Usamos titulo + id_lattes para identificar unicamente quem já deu match
ids_sucesso_exato = df_match_exato['titulo'] + df_match_exato['id_lattes']
mascara_nao_encontrados = ~(df_bib_trab_congresso['titulo'] + df_bib_trab_congresso['id_lattes']).isin(ids_sucesso_exato)

df_sem_match_exato = df_bib_trab_congresso[mascara_nao_encontrados].copy()

print("4. Executando Busca por Sigla (com proteção de palavra inteira)...")
# Cria um dicionário para busca rápida: { 'AAAI': {'Nome do evento': '...', 'Estrato': 'A1'} }
df_google_siglas_unicas = df_google_raw.dropna(subset=['Sigla']).drop_duplicates(subset=['Sigla'])
dict_siglas = df_google_siglas_unicas.set_index('Sigla')[['Nome do evento', 'Estrato']].to_dict('index')

# Lista de todas as siglas válidas para procurar
lista_siglas = list(dict_siglas.keys())

def buscar_sigla_no_texto(texto):
    if pd.isna(texto) or texto == 'NAN':
        return None
        
    for sigla in lista_siglas:
        if sigla != "":
            # \b significa 'fronteira de palavra'. Garante que 'SAC' só dê match em ' SAC ' e não em 'TRANSACTIONS'
            padrao = r'\b' + re.escape(sigla) + r'\b'
            if re.search(padrao, texto):
                return sigla
    return None

# Aplica a função de busca no nome do evento dos que sobraram
df_sem_match_exato['sigla_encontrada'] = df_sem_match_exato['evento_limpo'].apply(buscar_sigla_no_texto)

# Separa quem teve sucesso na busca por sigla
df_sucesso_sigla = df_sem_match_exato[df_sem_match_exato['sigla_encontrada'].notna()].copy()

# Traz os dados (Nome e Estrato) usando o mapeamento do dicionário
df_sucesso_sigla['Nome do evento'] = df_sucesso_sigla['sigla_encontrada'].apply(lambda x: dict_siglas[x]['Nome do evento'])
df_sucesso_sigla['Estrato'] = df_sucesso_sigla['sigla_encontrada'].apply(lambda x: dict_siglas[x]['Estrato'])
df_sucesso_sigla['Sigla'] = df_sucesso_sigla['sigla_encontrada']
df_sucesso_sigla['tipo_match'] = 'Por Sigla'

# Limpa colunas auxiliares
df_sucesso_sigla.drop(columns=['sigla_encontrada'], inplace=True)

print("5. Consolidando os que falharam em ambas as tentativas...")
df_falhas_totais = df_sem_match_exato[df_sem_match_exato['sigla_encontrada'].isna()].copy()
df_falhas_totais.drop(columns=['sigla_encontrada'], inplace=True)

# Preenche os vazios de quem não teve match (pode ajustar para 'A8' dependendo da sua regra de negócio)
df_falhas_totais['Nome do evento'] = pd.NA
df_falhas_totais['Estrato'] = 'A8' 
df_falhas_totais['Sigla'] = pd.NA
df_falhas_totais['tipo_match'] = 'Sem Match'

print("6. Empilhando tudo na base final...")
# Junta os 3 blocos: Sucesso Exato, Sucesso Sigla e Falhas
df_artigos_congresso_final = pd.concat([df_match_exato, df_sucesso_sigla, df_falhas_totais], ignore_index=True)

# Remove a coluna temporária de limpeza
df_artigos_congresso_final.drop(columns=['evento_limpo'], inplace=True)

# ==========================================
# RESUMO DOS RESULTADOS
# ==========================================
total_originais = len(df_bib_trab_congresso)
qtd_exato = len(df_match_exato)
qtd_sigla = len(df_sucesso_sigla)
qtd_falhas = len(df_falhas_totais)

print(f"\n--- 📊 RELATÓRIO DE CRUZAMENTO DE EVENTOS ---")
print(f"Total de Artigos (Lattes): {total_originais}")
print(f"✅ Match Exato (Nome): {qtd_exato} ({round((qtd_exato/total_originais)*100, 1)}%)")
print(f"✅ Match por Sigla: {qtd_sigla} ({round((qtd_sigla/total_originais)*100, 1)}%)")
print(f"❌ Sem Match: {qtd_falhas} ({round((qtd_falhas/total_originais)*100, 1)}%)")

# Exibe uma amostra dos que deram match por sigla para validar a qualidade da lógica
display(df_artigos_congresso_final[df_artigos_congresso_final['tipo_match'] == 'Por Sigla'][['evento', 'Sigla', 'Nome do evento', 'Estrato']].head())

1. Preparando os dados para o cruzamento...
2. Tentando Match Exato pelo Nome do Evento...
3. Isolando os artigos não encontrados para a busca por Sigla...
4. Executando Busca por Sigla (com proteção de palavra inteira)...
5. Consolidando os que falharam em ambas as tentativas...
6. Empilhando tudo na base final...

--- 📊 RELATÓRIO DE CRUZAMENTO DE EVENTOS ---
Total de Artigos (Lattes): 4461
✅ Match Exato (Nome): 377 (8.5%)
✅ Match por Sigla: 1719 (38.5%)
❌ Sem Match: 2365 (53.0%)


,evento,Sigla,Nome do evento,Estrato
377,CNMAC 2025,CNMAC,CONGRESSO NACIONAL DE MATEMÁTICA APLICADA E CO...,A8
378,BRACIS - BRAZILIAN CONFERENCE ON INTELLIGENT S...,BRACIS,BRAZILIAN CONFERENCE ON INTELLIGENT SYSTEMS,A3
379,THE 28TH INTERNATIONAL COMPUTING AND COMBINATO...,COCOON,INTERNATIONAL COMPUTING AND COMBINATORICS CONF...,A6
380,XI LATIN AND AMERICAN ALGORITHMS,LATIN,LATIN AMERICAN THEORETICAL INFORMATICS SYMPOSIUM,A3
381,XI LATIN AND AMERICAN ALGORITHMS,LATIN,LATIN AMERICAN THEORETICAL INFORMATICS SYMPOSIUM,A3


In [34]:
import pandas as pd
import re

print("1. Preparando dados e ordenando por tamanho do nome...")
df_bib_trab_congresso['evento_limpo'] = df_bib_trab_congresso['evento'].astype(str).str.upper().str.strip()
df_google_raw['Nome do evento'] = df_google_raw['Nome do evento'].astype(str).str.upper().str.strip()
df_google_raw['Sigla'] = df_google_raw['Sigla'].astype(str).str.upper().str.strip()

# O truque de ouro: ordenar os eventos do nome maior para o menor.
# Evita que um evento de nome curto "roube" o match de um evento mais específico.
df_google_raw['tamanho_nome'] = df_google_raw['Nome do evento'].str.len()
df_google_raw = df_google_raw.sort_values(by='tamanho_nome', ascending=False)

# Transforma a base do Google em uma lista de dicionários para a busca ser ultrarrápida
lista_google = df_google_raw.to_dict('records')

print("2. Aplicando a lógica de match (Nome Contido -> Sigla)...")

def encontrar_melhor_match(evento_lattes):
    if pd.isna(evento_lattes) or evento_lattes == 'NAN':
        return pd.NA, pd.NA, 'A8', 'Sem Match'

    # Tentativa 1: Verifica se o Nome Oficial da tabela Google está CONTIDO no nome digitado no Lattes
    # Como a lista está ordenada por tamanho, ele sempre pega o match mais completo primeiro.
    for google in lista_google:
        nome_oficial = google['Nome do evento']
        if pd.notna(nome_oficial) and nome_oficial != 'NAN' and nome_oficial != "":
            if nome_oficial in evento_lattes:
                return google['Sigla'], google['Nome do evento'], google['Estrato'], 'Por Nome (Contido)'

    # Tentativa 2: Se falhar no nome, busca a Sigla protegida por limites de palavra (\b)
    for google in lista_google:
        sigla = google['Sigla']
        if pd.notna(sigla) and sigla != 'NAN' and sigla != "":
            padrao = r'\b' + re.escape(sigla) + r'\b'
            if re.search(padrao, evento_lattes):
                return google['Sigla'], google['Nome do evento'], google['Estrato'], 'Por Sigla'

    return pd.NA, pd.NA, 'A8', 'Sem Match'

print("Isso pode levar alguns segundos...")
# Aplica a função de busca
resultados = df_bib_trab_congresso['evento_limpo'].apply(encontrar_melhor_match)

print("3. Consolidando base final...")
df_artigos_congresso_final = df_bib_trab_congresso.copy()

# Extraindo os resultados da função para suas respectivas colunas
df_artigos_congresso_final['Sigla'] = [res[0] for res in resultados]
df_artigos_congresso_final['Nome do evento'] = [res[1] for res in resultados]
df_artigos_congresso_final['Estrato'] = [res[2] for res in resultados]
df_artigos_congresso_final['tipo_match'] = [res[3] for res in resultados]

# Remove colunas auxiliares de limpeza
df_artigos_congresso_final.drop(columns=['evento_limpo'], inplace=True)

# ==========================================
# RESUMO DOS RESULTADOS
# ==========================================
total_originais = len(df_artigos_congresso_final)
qtd_nome = len(df_artigos_congresso_final[df_artigos_congresso_final['tipo_match'] == 'Por Nome (Contido)'])
qtd_sigla = len(df_artigos_congresso_final[df_artigos_congresso_final['tipo_match'] == 'Por Sigla'])
qtd_falhas = len(df_artigos_congresso_final[df_artigos_congresso_final['tipo_match'] == 'Sem Match'])

print(f"\n--- 📊 RELATÓRIO DE CRUZAMENTO DE EVENTOS ---")
print(f"Total de Artigos (Lattes): {total_originais}")
print(f"✅ Match por Nome (Contido/Exato): {qtd_nome} ({round((qtd_nome/total_originais)*100, 1)}%)")
print(f"✅ Match por Sigla: {qtd_sigla} ({round((qtd_sigla/total_originais)*100, 1)}%)")
print(f"❌ Sem Match: {qtd_falhas} ({round((qtd_falhas/total_originais)*100, 1)}%)")

# Exibe uma amostra dos matches para você validar
display(df_artigos_congresso_final[df_artigos_congresso_final['tipo_match'] != 'Sem Match'][['evento', 'Nome do evento', 'Estrato', 'tipo_match']].sample(5))

1. Preparando dados e ordenando por tamanho do nome...
2. Aplicando a lógica de match (Nome Contido -> Sigla)...
Isso pode levar alguns segundos...
3. Consolidando base final...

--- 📊 RELATÓRIO DE CRUZAMENTO DE EVENTOS ---
Total de Artigos (Lattes): 4461
✅ Match por Nome (Contido/Exato): 1732 (38.8%)
✅ Match por Sigla: 910 (20.4%)
❌ Sem Match: 1819 (40.8%)


,evento,Nome do evento,Estrato,tipo_match
1228,2019 IEEE SYMPOSIUM ON COMPUTERS AND COMMUNICA...,INTERNATIONAL SYMPOSIUM ON COMPUTERS AND COMMU...,A2,Por Sigla
4068,XXXVIII SIMPÓSIO BRASILEIRO DE PESQUISA OPERAC...,SIMPÓSIO BRASILEIRO DE PESQUISA OPERACIONAL,A4,Por Nome (Contido)
418,XIX BRAZILIAN SYMPOSIUM ON COMPUTER GAMES AND ...,SIMPÓSIO BRASILEIRO DE JOGOS E ENTRETENIMENTO ...,A4,Por Sigla
4044,XL SIMPÓSIO BRASILEIRO DE PESQUISA OPERACIONAL...,SIMPÓSIO BRASILEIRO DE PESQUISA OPERACIONAL,A4,Por Nome (Contido)
3301,31ST INTERNATIONAL CONFERENCE ON SOFTWARE ENGI...,INTERNATIONAL CONFERENCE ON SOFTWARE ENGINEERING,A1,Por Nome (Contido)


In [35]:
df_artigos_congresso_final.head()
print(df_artigos_congresso_final.info())

<class 'pandas.DataFrame'>
RangeIndex: 4461 entries, 0 to 4460
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   titulo          4461 non-null   str  
 1   ano             4461 non-null   Int64
 2   doi             1435 non-null   str  
 3   autores         4461 non-null   str  
 4   evento          4460 non-null   str  
 5   cidade          0 non-null      str  
 6   paginas         3057 non-null   str  
 7   isbn            0 non-null      str  
 8   id_lattes       4461 non-null   str  
 9   Sigla           2642 non-null   str  
 10  Nome do evento  2642 non-null   str  
 11  Estrato         4461 non-null   str  
 12  tipo_match      4461 non-null   str  
dtypes: Int64(1), str(12)
memory usage: 1.7 MB
None


In [36]:
import pandas as pd

print("Padronizando os nomes das colunas de eventos...")

# Dicionário com o mapeamento "Nome Antigo" : "Nome Novo"
mapeamento_colunas_eventos = {
    'titulo': 'titulo_artigo',
    'evento': 'titulo_evento_lattes',
    'Sigla': 'sigla_evento_google',
    'Nome do evento': 'titulo_evento_google',
    'Estrato': 'estrato'
}

# Aplica a renomeação diretamente no dataframe
df_artigos_congresso_final.rename(columns=mapeamento_colunas_eventos, inplace=True)

print("Colunas renomeadas com sucesso! Nova estrutura:")
df_artigos_congresso_final.info()

Padronizando os nomes das colunas de eventos...
Colunas renomeadas com sucesso! Nova estrutura:
<class 'pandas.DataFrame'>
RangeIndex: 4461 entries, 0 to 4460
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   titulo_artigo         4461 non-null   str  
 1   ano                   4461 non-null   Int64
 2   doi                   1435 non-null   str  
 3   autores               4461 non-null   str  
 4   titulo_evento_lattes  4460 non-null   str  
 5   cidade                0 non-null      str  
 6   paginas               3057 non-null   str  
 7   isbn                  0 non-null      str  
 8   id_lattes             4461 non-null   str  
 9   sigla_evento_google   2642 non-null   str  
 10  titulo_evento_google  2642 non-null   str  
 11  estrato               4461 non-null   str  
 12  tipo_match            4461 non-null   str  
dtypes: Int64(1), str(12)
memory usage: 1.7 MB


In [37]:
print("Removendo as colunas 'cidade' e 'isbn'...")

# O parâmetro errors='ignore' é uma trava de segurança. 
# Se você rodar a célula duas vezes sem querer, ele não vai dar erro reclamando que a coluna já sumiu.
df_artigos_congresso_final.drop(columns=['cidade', 'isbn'], inplace=True, errors='ignore')

print("Colunas removidas com sucesso! Estrutura atualizada:")
df_artigos_congresso_final.info()

Removendo as colunas 'cidade' e 'isbn'...
Colunas removidas com sucesso! Estrutura atualizada:
<class 'pandas.DataFrame'>
RangeIndex: 4461 entries, 0 to 4460
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   titulo_artigo         4461 non-null   str  
 1   ano                   4461 non-null   Int64
 2   doi                   1435 non-null   str  
 3   autores               4461 non-null   str  
 4   titulo_evento_lattes  4460 non-null   str  
 5   paginas               3057 non-null   str  
 6   id_lattes             4461 non-null   str  
 7   sigla_evento_google   2642 non-null   str  
 8   titulo_evento_google  2642 non-null   str  
 9   estrato               4461 non-null   str  
 10  tipo_match            4461 non-null   str  
dtypes: Int64(1), str(10)
memory usage: 1.7 MB


## Conectando com DuckDB

In [38]:
import duckdb

print("Iniciando persistência no DuckDB com as tabelas de Artigos e Orientações...")
con = duckdb.connect('pesquisadores.duckdb')

# ==========================================
# 1. CRIAÇÃO DOS SCHEMAS E SEQUÊNCIAS
# ==========================================

# Tabela Mãe: Professores
query_cria_pessoas = """
CREATE TABLE IF NOT EXISTS tb_professores (
    id_lattes VARCHAR PRIMARY KEY,
    nome_completo VARCHAR,
    nome_citacoes VARCHAR,
    sexo VARCHAR,
    rotulo VARCHAR,
    periodo VARCHAR,
    bolsa_produtividade VARCHAR,
    endereco_profissional VARCHAR,
    atualizacao_cv TIMESTAMP,
    url VARCHAR,
    texto_resumo VARCHAR
);
"""
con.execute(query_cria_pessoas)

# Sequências para os IDs automáticos das três tabelas filhas
con.execute("CREATE SEQUENCE IF NOT EXISTS seq_id_artigo_periodico;")
con.execute("CREATE SEQUENCE IF NOT EXISTS seq_id_artigo_conferencia;")
con.execute("CREATE SEQUENCE IF NOT EXISTS seq_id_orientacao;")

# Tabela Filha 1: Artigos de Periódicos (Scopus)
query_cria_periodicos = """
CREATE TABLE IF NOT EXISTS tb_artigo_periodico (
    id_artigo_periodico INTEGER PRIMARY KEY DEFAULT nextval('seq_id_artigo_periodico'),
    id_lattes VARCHAR,
    titulo_artigo VARCHAR NOT NULL,
    titulo_revista_lattes VARCHAR,
    ano_pub INTEGER,
    match_adequado BOOLEAN,
    id_scopus VARCHAR,
    titulo_revista_scopus VARCHAR,
    maior_percentil INTEGER,
    codigo_area_maior_percentil VARCHAR,
    area_maior_percentil VARCHAR,
    issn VARCHAR,
    computation_area BOOLEAN,
    FOREIGN KEY (id_lattes) REFERENCES tb_professores(id_lattes)
);
"""
con.execute(query_cria_periodicos)

# Tabela Filha 2: Artigos de Conferências/Congressos (Google)
query_cria_conferencias = """
CREATE TABLE IF NOT EXISTS tb_artigo_conferencia (
    id_artigo_conferencia INTEGER PRIMARY KEY DEFAULT nextval('seq_id_artigo_conferencia'),
    id_lattes VARCHAR,
    titulo_artigo VARCHAR NOT NULL,
    ano INTEGER,
    doi VARCHAR,
    autores VARCHAR,
    titulo_evento_lattes VARCHAR,
    paginas VARCHAR,
    sigla_evento_google VARCHAR,
    titulo_evento_google VARCHAR,
    estrato VARCHAR,
    tipo_match VARCHAR,
    FOREIGN KEY (id_lattes) REFERENCES tb_professores(id_lattes)
);
"""
con.execute(query_cria_conferencias)

# Tabela Filha 3: Orientações (NOVA)
query_cria_orientacoes = """
CREATE TABLE IF NOT EXISTS tb_orientacoes (
    id_orientacao INTEGER PRIMARY KEY DEFAULT nextval('seq_id_orientacao'),
    id_lattes VARCHAR,
    titulo_trabalho VARCHAR,
    ano_inicio INTEGER,
    orientando VARCHAR,
    tipo_trabalho VARCHAR,
    instituicao VARCHAR,
    curso VARCHAR,
    status VARCHAR,
    nivel VARCHAR,
    ano_conclusao INTEGER,
    FOREIGN KEY (id_lattes) REFERENCES tb_professores(id_lattes)
);
"""
con.execute(query_cria_orientacoes)

print("Tabelas criadas com sucesso (ou já existentes).")

# ==========================================
# 2. INSERÇÃO DOS DADOS (Carga via Pandas)
# ==========================================
print("Limpando dados antigos (Filhas primeiro, Mãe depois)...")
# Apagar as 3 filhas antes da mãe para não violar a integridade relacional
con.execute("DELETE FROM tb_artigo_periodico")
con.execute("DELETE FROM tb_artigo_conferencia")
con.execute("DELETE FROM tb_orientacoes")
con.execute("DELETE FROM tb_professores")

print("Inserindo novos dados a partir dos DataFrames Pandas...")

# Inserção da Tabela Mãe (Professores)
if not df_pessoas.empty:
    con.execute("INSERT INTO tb_professores SELECT * FROM df_pessoas")

# Inserção da Tabela Filha 1 (Artigos Periódicos)
if not df_artigos_final.empty:
    con.execute("""
        INSERT INTO tb_artigo_periodico (
            id_lattes, titulo_artigo, titulo_revista_lattes, ano_pub, 
            match_adequado, id_scopus, titulo_revista_scopus, maior_percentil, 
            codigo_area_maior_percentil, area_maior_percentil, issn, computation_area
        )
        SELECT 
            id_lattes, titulo_artigo, titulo_revista_lattes, ano_pub, 
            match_adequado, id_scopus, titulo_revista_scopus, maior_percentil, 
            codigo_area_maior_percentil, area_maior_percentil, issn, computation_area 
        FROM df_artigos_final
    """)

# Inserção da Tabela Filha 2 (Artigos Conferência)
if not df_artigos_congresso_final.empty:
    con.execute("""
        INSERT INTO tb_artigo_conferencia (
            id_lattes, titulo_artigo, ano, doi, autores, 
            titulo_evento_lattes, paginas, sigla_evento_google, 
            titulo_evento_google, estrato, tipo_match
        )
        SELECT 
            id_lattes, titulo_artigo, ano, doi, autores, 
            titulo_evento_lattes, paginas, sigla_evento_google, 
            titulo_evento_google, estrato, tipo_match
        FROM df_artigos_congresso_final
    """)

# Inserção da Tabela Filha 3 (Orientações) - NOVA
if not df_orientacoes.empty:
    con.execute("""
        INSERT INTO tb_orientacoes (
            id_lattes, titulo_trabalho, ano_inicio, orientando, 
            tipo_trabalho, instituicao, curso, status, nivel, ano_conclusao
        )
        SELECT 
            id_lattes, titulo_trabalho, ano_inicio, orientando, 
            tipo_trabalho, instituicao, curso, status, nivel, ano_conclusao
        FROM df_orientacoes
    """)

con.close()
print("Processo finalizado! Banco 'pesquisadores.duckdb' atualizado com o schema completo.")

Iniciando persistência no DuckDB com as tabelas de Artigos e Orientações...
Tabelas criadas com sucesso (ou já existentes).
Limpando dados antigos (Filhas primeiro, Mãe depois)...
Inserindo novos dados a partir dos DataFrames Pandas...
Processo finalizado! Banco 'pesquisadores.duckdb' atualizado com o schema completo.
